# 🚀 Sketch2Shape AI - Colab GPU Inference Server (Stages 1 - 6)
This notebook powers the GPU-accelerated backend for Sketch2Shape AI:
- **Stage 1**: OpenCV Sketch Pre-Processing (CLAHE, Aspect-Ratio Padding, Auto-Inversion)
- **Stage 2**: Sketch Understanding (Google Gemini Vision API via Key 1)
- **Stage 3**: 2.5D Volumetric Concept Generation (Gemini Key 2 SVG + Solid Studio Clay Render)
- **Stage 4**: Foreground Isolation (rembg BiRefNet on GPU)
- **Stage 5A**: Fast 3D Reconstruction (TripoSR / SF3D — ~6 GB VRAM, 0.5s–12s)
- **Stage 5B**: Ultra-HD 3D Reconstruction (Hunyuan3D-2mini / TRELLIS — ~12 GB VRAM, 15s–28s)
- **Stage 6**: Mesh Validation, Watertight Manifolds & GLB/OBJ Export via Trimesh
- **Public Tunnel**: Exposes FastAPI endpoints via  or  for your React Three.js Frontend.

In [ ]:
# Step 1: Check GPU Runtime
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 2: Install Dependencies
!pip install -q fastapi uvicorn pydantic pyngrok rembg trimesh opencv-python-headless pillow httpx einops
# Install TripoSR for Stage 5A
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git
print("Dependencies installed successfully!")

In [ ]:
# Step 3: Configure Gemini API Keys & Ngrok Token
import os
# Key 1: Vision Understanding (paste your Gemini API Key here)
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")
# Key 2: 2.5D Volumetric Concept Generation
os.environ["GEMINI_25D_API_KEY"] = os.environ.get("GEMINI_25D_API_KEY", "YOUR_GEMINI_25D_API_KEY")

# (Optional) Add your free Ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # Leave empty if using localtunnel

In [ ]:
# Step 4: Write & Launch the Colab Backend Server
from colab_backend import start_colab_tunnel
start_colab_tunnel(port=8000, ngrok_token=NGROK_TOKEN if NGROK_TOKEN else None)

In [ ]:
# Step 5 (Alternative): If not using ngrok, expose via localtunnel
# Note: Run this cell if NGROK_TOKEN is empty
!npm install -g localtunnel
!npx localtunnel --port 8000